<a href="https://colab.research.google.com/github/CoderMakar/Plenki/blob/main/MOST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# DATABASE A — NBD/QC MOST
# Чистый полный pipeline для Google Colab
# ============================================================

import os
import sys
import subprocess
import zipfile
import warnings

warnings.filterwarnings("ignore")


# ============================================================
# 1. УСТАНОВКА ЗАВИСИМОСТЕЙ
# ============================================================

print("=== 1. Установка зависимостей ===")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "rdkit"],
    check=True
)

# xyz2mol нужен старому авторскому коду qmconf
XYZ2MOL_DIR = "/content/xyz2mol"

if not os.path.isdir(XYZ2MOL_DIR):
    subprocess.run(
        [
            "git",
            "clone",
            "-q",
            "https://github.com/jensengroup/xyz2mol.git",
            XYZ2MOL_DIR
        ],
        check=True
    )

print("Зависимости готовы.")


# ============================================================
# 2. СКАЧИВАНИЕ АРХИВА
# ============================================================

print("\n=== 2. Исходный архив ===")

ZIP_PATH = "/content/project.zip"

URL = (
    "https://sid.erda.dk/share_redirect/"
    "DeRV97z1Nz/"
    "virtual_screening_of_norbornadiene_based_"
    "molecular_solar_thermal_energy_storage_systems_"
    "using_a_genetic_algorithm.zip"
)

if not os.path.isfile(ZIP_PATH):

    print("Скачиваю архив...")

    subprocess.run(
        [
            "wget",
            "-q",
            "--show-progress",
            URL,
            "-O",
            ZIP_PATH
        ],
        check=True
    )

else:
    print("Архив уже загружен:", ZIP_PATH)


# ============================================================
# 3. РАСПАКОВКА
# ============================================================

print("\n=== 3. Распаковка ===")

EXTRACT_DIR = "/content/project"

if not os.path.isdir(EXTRACT_DIR):
    os.makedirs(EXTRACT_DIR, exist_ok=True)

    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)

    print("Архив распакован.")
else:
    print("Папка проекта уже существует.")


# ============================================================
# 4. ПУТИ ПРОЕКТА
# ============================================================

BASE = os.path.join(
    EXTRACT_DIR,
    "virtual_screening_of_norbornadiene_based_"
    "molecular_solar_thermal_energy_storage_systems_"
    "using_a_genetic_algorithm"
)

QMC = os.path.join(
    BASE,
    "dependencies",
    "tQMC",
    "QMC"
)

CALC_DIR = os.path.join(
    QMC,
    "calculator"
)

DATASET_DIR = os.path.join(
    BASE,
    "dataset"
)

RAW_PATH = os.path.join(
    DATASET_DIR,
    "result_abs.pkl"
)

READY_PATH = os.path.join(
    DATASET_DIR,
    "result_abs_700_nodublicates.pkl"
)


required_paths = {
    "QMC": QMC,
    "calculator": CALC_DIR,
    "RAW dataset": RAW_PATH,
    "READY dataset": READY_PATH
}

for name, path in required_paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{name} не найден:\n{path}"
        )

print("\nВсе исходные файлы найдены.")


# ============================================================
# 5. ПОДКЛЮЧЕНИЕ АВТОРСКИХ МОДУЛЕЙ
# ============================================================

print("\n=== 4. Подключение QMC ===")

# calculator должен быть Python-пакетом
init_file = os.path.join(
    CALC_DIR,
    "__init__.py"
)

if not os.path.exists(init_file):
    with open(init_file, "w"):
        pass


# Нельзя добавлять QMC/calculator напрямую,
# иначе calculator импортируется как calculator.py,
# а не как пакет calculator.*
sys.path = [
    p for p in sys.path
    if p != CALC_DIR
]

if QMC not in sys.path:
    sys.path.insert(0, QMC)

if "/content" not in sys.path:
    sys.path.insert(0, "/content")


# Очищаем возможные старые импорты
for module_name in list(sys.modules):

    if (
        module_name == "qmconf"
        or module_name == "calculator"
        or module_name.startswith("calculator.")
    ):
        del sys.modules[module_name]


from qmconf import QMConf
import calculator.xtb

print("QMC подключён.")
print("calculator.xtb:", calculator.xtb.__file__)


# ============================================================
# 6. ЗАГРУЗКА DATASET
# ============================================================

print("\n=== 5. Загрузка данных ===")

import pandas as pd
import numpy as np

df_raw = pd.read_pickle(
    RAW_PATH
)

df_ready = pd.read_pickle(
    READY_PATH
)


print(
    "RAW:",
    df_raw.shape
)

print(
    "READY:",
    df_ready.shape
)

print(
    "READY columns:",
    df_ready.columns.tolist()
)


# Проверки готовой авторской выборки
assert len(df_ready) == 33706
assert df_ready["smiles"].nunique() == 33706
assert df_ready.isna().sum().sum() == 0

print(
    "READY dataset корректен:",
    "33706 уникальных записей."
)


# ============================================================
# 7. СОЗДАНИЕ comp_name В RAW
# ============================================================

print("\n=== 6. Связь READY ↔ RAW ===")

df_raw = df_raw.copy()

df_raw["comp_name"] = (
    df_raw["prod"]
    .apply(
        lambda obj:
        str(obj.label.split("_p")[0])
    )
)

df_raw["raw_index"] = df_raw.index


print(
    "RAW строк:",
    len(df_raw)
)

print(
    "RAW уникальных comp_name:",
    df_raw["comp_name"].nunique()
)


assert df_raw["comp_name"].is_unique


# ============================================================
# 8. MERGE ПО comp_name
# ============================================================

raw_map = df_raw[
    [
        "raw_index",
        "comp_name",
        "reac",
        "prod"
    ]
].copy()


ready_map = df_ready[
    [
        "comp_name",
        "smiles",
        "storage",
        "tbr",
        "absorp"
    ]
].copy()


ready_map = ready_map.rename(
    columns={
        "smiles": "qc_smiles_raw"
    }
)


matched = ready_map.merge(
    raw_map,
    on="comp_name",
    how="left",
    validate="one_to_one"
)


print(
    "Сопоставлено:",
    matched["raw_index"].notna().sum(),
    "/",
    len(matched)
)


assert matched["raw_index"].notna().all()


# ============================================================
# 9. ИЗВЛЕЧЕНИЕ NBD И QC SMILES
# ============================================================

print("\n=== 7. Подготовка структур ===")

from rdkit import Chem
from rdkit.Chem import Descriptors


def qmconf_to_smiles(obj):
    """
    QMConf -> canonical SMILES.
    Используется для исходного NBD (reac).
    """

    try:

        mol = obj.rdkit_mol

        if mol is None:
            mol = obj.get_rdkit_mol()

        if mol is None:
            return None

        mol = Chem.RemoveHs(mol)

        return Chem.MolToSmiles(
            mol,
            canonical=True,
            isomericSmiles=False
        )

    except Exception:
        return None


def normalize_smiles(smiles):
    """
    SMILES -> canonical SMILES.
    """

    try:

        mol = Chem.MolFromSmiles(
            smiles
        )

        if mol is None:
            return None

        mol = Chem.RemoveHs(mol)

        return Chem.MolToSmiles(
            mol,
            canonical=True,
            isomericSmiles=False
        )

    except Exception:
        return None


# NBD из RAW reac
matched["nbd_smiles"] = (
    matched["reac"]
    .apply(qmconf_to_smiles)
)


# QC из GCN-ready набора авторов
matched["qc_smiles"] = (
    matched["qc_smiles_raw"]
    .apply(normalize_smiles)
)


print(
    "NBD получено:",
    matched["nbd_smiles"].notna().sum()
)

print(
    "QC получено:",
    matched["qc_smiles"].notna().sum()
)

print(
    "Уникальных NBD:",
    matched["nbd_smiles"].nunique()
)

print(
    "Уникальных QC:",
    matched["qc_smiles"].nunique()
)


assert matched["nbd_smiles"].notna().all()
assert matched["qc_smiles"].notna().all()

assert (
    matched["nbd_smiles"].nunique()
    == 33706
)

assert (
    matched["qc_smiles"].nunique()
    == 33706
)


# ============================================================
# 10. МОЛЯРНАЯ МАССА
# ============================================================

print("\n=== 8. Расчёт энергетической плотности ===")


def get_mol_weight(smiles):

    mol = Chem.MolFromSmiles(
        smiles
    )

    if mol is None:
        return np.nan

    return Descriptors.MolWt(
        mol
    )


matched["mol_weight_g_mol"] = (
    matched["nbd_smiles"]
    .apply(get_mol_weight)
)


# ============================================================
# 11. ENERGY DENSITY
#
# storage          kJ/mol
# molecular weight g/mol
#
# kJ/mol ÷ g/mol
# = kJ/g
# = MJ/kg
# ============================================================

matched["energy_density_MJ_kg"] = (
    matched["storage"]
    /
    matched["mol_weight_g_mol"]
)


# ============================================================
# 12. DATABASE A
# ============================================================

print("\n=== 9. Формирование Database A ===")


database_A_final = pd.DataFrame({

    "comp_name":
        matched["comp_name"],

    "nbd_smiles":
        matched["nbd_smiles"],

    "qc_smiles":
        matched["qc_smiles"],

    "storage_kJ_mol":
        matched["storage"],

    "energy_density_MJ_kg":
        matched["energy_density_MJ_kg"],

    "tbr_kJ_mol":
        matched["tbr"],

    "absorption_nm":
        matched["absorp"]
})


# ============================================================
# 13. ПРОВЕРКА DATABASE A
# ============================================================

print(
    "Размер:",
    database_A_final.shape
)

print(
    "Уникальных NBD:",
    database_A_final[
        "nbd_smiles"
    ].nunique()
)

print(
    "Уникальных QC:",
    database_A_final[
        "qc_smiles"
    ].nunique()
)

print("\nПропуски:")

print(
    database_A_final
    .isna()
    .sum()
)


assert database_A_final.shape == (
    33706,
    7
)

assert (
    database_A_final
    .isna()
    .sum()
    .sum()
    == 0
)

assert (
    database_A_final[
        "nbd_smiles"
    ].is_unique
)

assert (
    database_A_final[
        "qc_smiles"
    ].is_unique
)


# ============================================================
# 14. СТАТИСТИКА
# ============================================================

print(
    "\n=== Статистика свойств ==="
)

display(

    database_A_final[
        [
            "storage_kJ_mol",
            "energy_density_MJ_kg",
            "tbr_kJ_mol",
            "absorption_nm"
        ]
    ]
    .describe()

)


print(
    "\n=== Первые молекулы ==="
)

display(
    database_A_final.head(10)
)


# ============================================================
# 15. DATASET ДЛЯ ГЕНЕРАТОРА
# ============================================================

generator_corpus = (

    database_A_final[
        [
            "comp_name",
            "nbd_smiles"
        ]
    ]

    .rename(
        columns={
            "nbd_smiles":
            "smiles"
        }
    )

    .copy()
)


# ============================================================
# 16. DATASET ДЛЯ MOST EXPERT
# ============================================================

most_expert_data = (

    database_A_final[
        [
            "comp_name",
            "nbd_smiles",
            "energy_density_MJ_kg",
            "tbr_kJ_mol"
        ]
    ]

    .rename(
        columns={
            "nbd_smiles":
            "smiles"
        }
    )

    .copy()
)


print(
    "\nGenerator:",
    generator_corpus.shape
)

print(
    "MOST expert:",
    most_expert_data.shape
)


# ============================================================
# 17. СОХРАНЕНИЕ
# ============================================================

OUT_A = (
    "/content/"
    "database_A_MOST_final.csv"
)

OUT_GENERATOR = (
    "/content/"
    "NBD_generator_corpus.csv"
)

OUT_EXPERT = (
    "/content/"
    "MOST_expert_dataset.csv"
)


database_A_final.to_csv(
    OUT_A,
    index=False
)

generator_corpus.to_csv(
    OUT_GENERATOR,
    index=False
)

most_expert_data.to_csv(
    OUT_EXPERT,
    index=False
)


# ============================================================
# 18. ИТОГ
# ============================================================

print(
    "\n======================================"
)

print(
    "DATABASE A ГОТОВА"
)

print(
    "======================================"
)

print(
    "Database A:",
    database_A_final.shape
)

print(
    "Generator:",
    generator_corpus.shape
)

print(
    "MOST expert:",
    most_expert_data.shape
)

print(
    "\nСохранено:"
)

print(
    OUT_A
)

print(
    OUT_GENERATOR
)

print(
    OUT_EXPERT
)

=== 1. Установка зависимостей ===
Зависимости готовы.

=== 2. Исходный архив ===
Скачиваю архив...

=== 3. Распаковка ===
Архив распакован.

Все исходные файлы найдены.

=== 4. Подключение QMC ===
QMC подключён.
calculator.xtb: /content/project/virtual_screening_of_norbornadiene_based_molecular_solar_thermal_energy_storage_systems_using_a_genetic_algorithm/dependencies/tQMC/QMC/calculator/xtb.py

=== 5. Загрузка данных ===
RAW: (55568, 7)
READY: (33706, 5)
READY columns: ['comp_name', 'smiles', 'storage', 'tbr', 'absorp']
READY dataset корректен: 33706 уникальных записей.

=== 6. Связь READY ↔ RAW ===
RAW строк: 55568
RAW уникальных comp_name: 55568
Сопоставлено: 33706 / 33706

=== 7. Подготовка структур ===
NBD получено: 33706
QC получено: 33706
Уникальных NBD: 33706
Уникальных QC: 33706

=== 8. Расчёт энергетической плотности ===

=== 9. Формирование Database A ===
Размер: (33706, 7)
Уникальных NBD: 33706
Уникальных QC: 33706

Пропуски:
comp_name               0
nbd_smiles           

,storage_kJ_mol,energy_density_MJ_kg,tbr_kJ_mol,absorption_nm
count,33706.000000,33706.000000,33706.000000,33706.000000
mean,23.961666,0.079843,212.928727,325.045559
std,8.584163,0.035754,31.323953,41.013722
min,0.099234,0.000310,65.437325,159.700000
25%,17.236745,0.060036,192.306838,297.600000
50%,23.369080,0.073757,210.299817,314.600000
75%,28.854381,0.088511,242.486067,344.400000
max,86.068099,0.569707,320.298085,643.500000



=== Первые молекулы ===


,comp_name,nbd_smiles,qc_smiles,storage_kJ_mol,energy_density_MJ_kg,tbr_kJ_mol,absorption_nm
0,A-0_A-0_A-0_A-0,C1=CC2C=CC1C2,C1C2C3C2C2C1C32,8.111205,0.088030,320.298085,159.7
1,A-1_A-0_A-0_A-0,FC1=CC2C=CC1C2,FC12C3CC4C(C41)C32,22.362574,0.203054,290.434249,177.7
2,A-2_A-0_A-0_A-0,FC(F)(F)C1=CC2C=CC1C2,FC(F)(F)C12C3CC4C(C41)C32,7.536407,0.047062,284.936736,167.8
3,A-3_A-0_A-0_A-0,N#CC1=CC2C=CC1C2,N#CC12C3CC4C(C41)C32,9.834636,0.083948,258.947101,220.8
4,A-4_A-0_A-0_A-0,O=[N+]([O-])C1=CC2C=CC1C2,O=[N+]([O-])C12C3CC4C(C41)C32,5.430744,0.039601,219.128675,295.2
5,A-5_A-0_A-0_A-0,O=CC1=CC2C=CC1C2,O=CC12C3CC4C(C41)C32,3.724758,0.031001,240.969550,239.4
6,A-6_A-0_A-0_A-0,O=C(O)C1=CC2C=CC1C2,O=C(O)C12C3CC4C(C41)C32,3.173696,0.023310,252.096535,232.3
7,A-7_A-0_A-0_A-0,CC(=O)C1=CC2C=CC1C2,CC(=O)C12C3CC4C(C41)C32,2.787060,0.020771,248.139052,294.6
8,A-8_A-0_A-0_A-0,NC(=O)C1=CC2C=CC1C2,NC(=O)C12C3CC4C(C41)C32,5.610543,0.041509,260.607473,238.2
9,A-9_A-0_A-0_A-0,CS(=O)(=O)C1=CC2C=CC1C2,CS(=O)(=O)C12C3CC4C(C41)C32,5.274700,0.030985,282.684282,230.3



Generator: (33706, 2)
MOST expert: (33706, 4)

DATABASE A ГОТОВА
Database A: (33706, 7)
Generator: (33706, 2)
MOST expert: (33706, 4)

Сохранено:
/content/database_A_MOST_final.csv
/content/NBD_generator_corpus.csv
/content/MOST_expert_dataset.csv
